# Seemingly Unrelated Regressions (SUR)

This notebook introduces the **SUR estimator** for panel data, treating each entity as an equation in a system with cross-equation correlated errors.

**Key references:**
- Zellner (1962) — original SUR estimator
- Zellner & Huang (1962) — efficiency properties
- McElroy (1977) — system R-squared
- Breusch & Pagan (1980) — independence test

**Topics covered:**
1. Why cross-equation correlation matters
2. Basic SUR with Grunfeld data
3. OLS vs SUR: standard error comparison
4. Breusch-Pagan independence test
5. Heatmap of the Sigma matrix
6. Heterogeneous formulas per entity
7. Iterated SUR (ISUR) vs non-iterated
8. Homogeneous SUR vs unrestricted
9. Application: credit risk pools (simulated)
10. Exercises

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import panelbox as pb

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

---
## 1. Why Cross-Equation Correlation Matters

In standard panel models (FE, RE), we estimate a single equation:

$$y_{it} = \alpha_i + \beta' x_{it} + \varepsilon_{it}$$

But sometimes the **errors across entities are correlated**. For example, in the Grunfeld investment data, a macroeconomic shock may simultaneously affect investment decisions across multiple firms.

The SUR estimator treats each entity $i$ as a separate equation:

$$y_i = X_i \beta_i + \varepsilon_i, \quad i = 1, \ldots, N$$

and exploits the cross-equation covariance structure $\Sigma = E[\varepsilon_i \varepsilon_j']$ to produce more **efficient** estimates than equation-by-equation OLS.

**When does SUR help?**
- Errors are correlated across equations ($\sigma_{ij} \neq 0$)
- Regressors differ across equations (different $X_i$)

If errors are uncorrelated OR all equations have the same regressors, SUR reduces to OLS — there is no efficiency gain.

---
## 2. Basic SUR with Grunfeld Data

The Grunfeld (1958) dataset contains investment data for 10 US firms over 20 years. Each firm is an equation in our SUR system.

In [ ]:
data = pb.load_grunfeld()
print(f"Panel: {data['firm'].nunique()} firms x {data['year'].nunique()} years = {len(data)} obs")
data.head(10)

In [ ]:
# Fit SUR: each firm is an equation
sur = pb.SUR('invest ~ value + capital', data, 'firm', 'year')
res = sur.fit()

# System-level summary
print(res.system_summary())

The system summary shows:
- The cross-equation **covariance** and **correlation** matrices
- The **Breusch-Pagan test** for independence
- Per-equation **R-squared** values
- **Efficiency gain** ratios (SUR SE / OLS SE): values below 1.0 mean SUR is more precise

In [ ]:
# Equation-level detail for a specific firm
entities = list(res.entity_params.keys())
print(res.equation_summary(entities[0]))

---
## 3. OLS vs SUR: Standard Error Comparison

Let's compare the standard errors from entity-by-entity OLS with SUR estimates.

In [ ]:
# Build comparison table
rows = []
for entity in entities:
    for var in res.entity_params[entity].index:
        ols_se = res.ols_params[entity][var]  # OLS coef (for reference)
        sur_coef = res.entity_params[entity][var]
        sur_se = res.entity_std_errors[entity][var]
        gain = res.efficiency_gain[entity][var]
        rows.append({
            'Entity': entity, 'Variable': var,
            'SUR Coef': sur_coef, 'SUR SE': sur_se,
            'Efficiency Gain': gain
        })

comp_df = pd.DataFrame(rows)
print("Efficiency Gain < 1 means SUR is more efficient than OLS")
print(f"Mean efficiency gain: {comp_df['Efficiency Gain'].mean():.4f}")
print(f"Min efficiency gain:  {comp_df['Efficiency Gain'].min():.4f}")
print(f"Max efficiency gain:  {comp_df['Efficiency Gain'].max():.4f}")

In [ ]:
# Visualize efficiency gains
fig, ax = plt.subplots(figsize=(12, 6))

# One bar per entity-variable combination
gains_value = [res.efficiency_gain[e].get('value', np.nan) for e in entities]
gains_capital = [res.efficiency_gain[e].get('capital', np.nan) for e in entities]

x = np.arange(len(entities))
width = 0.35

bars1 = ax.bar(x - width/2, gains_value, width, label='value', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, gains_capital, width, label='capital', color='coral', alpha=0.8)

ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='No gain (=OLS)')
ax.set_xlabel('Entity (Firm)')
ax.set_ylabel('SE_SUR / SE_OLS')
ax.set_title('SUR Efficiency Gain by Entity and Variable')
ax.set_xticks(x)
ax.set_xticklabels([str(e) for e in entities], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Breusch-Pagan Independence Test

The **Breusch-Pagan (1980) LM test** tests whether the off-diagonal elements of $\Sigma$ are jointly zero:

- $H_0$: $\sigma_{ij} = 0$ for all $i \neq j$ (errors are independent across equations)
- $H_1$: At least one $\sigma_{ij} \neq 0$

$$LM = T \sum_{i=2}^{N} \sum_{j=1}^{i-1} r_{ij}^2 \sim \chi^2\left(\frac{N(N-1)}{2}\right)$$

where $r_{ij}$ are the sample correlations of residuals.

In [ ]:
bp = res.bp_independence_test

print("Breusch-Pagan Independence Test")
print("=" * 40)
print(f"LM statistic: {bp['statistic']:.4f}")
print(f"p-value:      {bp['pvalue']:.6f}")
print(f"df:           {bp['df']}")
print()
if bp['pvalue'] < 0.05:
    print("Reject H0: Cross-equation errors ARE correlated.")
    print("SUR is expected to be more efficient than OLS.")
else:
    print("Fail to reject H0: No evidence of cross-equation correlation.")
    print("SUR may not offer gains over OLS.")

---
## 5. Heatmap of the Sigma Matrix

The cross-equation correlation matrix reveals which entities have correlated errors.

In [ ]:
# Built-in heatmap
fig, ax = plt.subplots(figsize=(9, 7))
res.plot_correlation_matrix(ax=ax)
ax.set_title('Cross-Equation Correlation Matrix (Grunfeld)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Sigma covariance matrix as DataFrame
sigma_df = pd.DataFrame(
    res.sigma_matrix,
    index=[str(e) for e in entities],
    columns=[str(e) for e in entities]
)
print("Cross-equation Covariance Matrix (Sigma):")
sigma_df.round(2)

---
## 6. Heterogeneous Formulas per Entity

A unique feature of SUR is that each equation can have **different regressors**. This is where SUR gains the most efficiency over OLS.

Let's specify different formulas for different firms.

In [ ]:
# Use a subset of firms with different formulas
firms = sorted(data['firm'].unique())[:4]
sub_data = data[data['firm'].isin(firms)].copy()

formulas = {
    firms[0]: 'invest ~ value + capital',
    firms[1]: 'invest ~ value',              # only market value
    firms[2]: 'invest ~ capital',             # only capital stock
    firms[3]: 'invest ~ value + capital',
}

print("Heterogeneous formulas:")
for firm, f in formulas.items():
    print(f"  Firm {firm}: {f}")

In [ ]:
sur_het = pb.SUR(formulas, sub_data, 'firm', 'year')
res_het = sur_het.fit()

print(res_het.system_summary())

In [ ]:
# Show per-equation coefficients
for firm in firms:
    params = res_het.entity_params[firm]
    se = res_het.entity_std_errors[firm]
    print(f"\nFirm {firm}:")
    for var in params.index:
        print(f"  {var:<12} coef={params[var]:>9.4f}  se={se[var]:>8.4f}")

With different regressors across equations, SUR borrows information through the correlated errors to improve efficiency. This is the classic setting where SUR shines.

---
## 7. Iterated SUR (ISUR) vs Non-Iterated

The standard SUR uses a **two-step** procedure:
1. OLS to estimate residuals and $\hat{\Sigma}$
2. FGLS using $\hat{\Sigma}$

**Iterated SUR (ISUR)** repeats this process:
1. Estimate $\beta$ via FGLS
2. Update $\hat{\Sigma}$ from new residuals
3. Re-estimate $\beta$
4. Repeat until convergence

Under normality, ISUR converges to the **Maximum Likelihood** estimator.

In [ ]:
# Non-iterated SUR
sur_base = pb.SUR('invest ~ value + capital', data, 'firm', 'year')
res_base = sur_base.fit()

# Iterated SUR
sur_iter = pb.SUR('invest ~ value + capital', data, 'firm', 'year',
                   iterate=True, max_iter=200, tol=1e-8)
res_iter = sur_iter.fit()

print(f"Non-iterated SUR: 1 iteration")
print(f"Iterated SUR:     {res_iter.n_iterations} iterations, converged={res_iter.converged}")
print(f"\nSystem R-squared:")
print(f"  Non-iterated: {res_base.system_rsquared:.6f}")
print(f"  Iterated:     {res_iter.system_rsquared:.6f}")

In [ ]:
# Compare coefficients: non-iterated vs iterated
print(f"{'Entity':<10} {'Variable':<12} {'SUR':>10} {'ISUR':>10} {'Diff':>10}")
print("-" * 55)
for entity in entities[:5]:  # show first 5 firms
    for var in res_base.entity_params[entity].index:
        b1 = res_base.entity_params[entity][var]
        b2 = res_iter.entity_params[entity][var]
        print(f"{str(entity):<10} {var:<12} {b1:>10.4f} {b2:>10.4f} {b2-b1:>10.4f}")

In practice, the differences between SUR and ISUR are usually small. ISUR may be preferred when you want MLE properties under normality.

---
## 8. Homogeneous SUR vs Unrestricted

**Homogeneous SUR** constrains all entities to share the same slope coefficients:

$$\beta_1 = \beta_2 = \cdots = \beta_N = \beta$$

This is more efficient if the restriction holds, but inconsistent if slopes truly differ.

We can test homogeneity using a **Likelihood Ratio test**.

In [ ]:
# Unrestricted SUR (different beta per entity)
sur_unr = pb.SUR('invest ~ value + capital', data, 'firm', 'year')
res_unr = sur_unr.fit()

# Homogeneous SUR (same beta for all entities)
sur_hom = pb.SUR('invest ~ value + capital', data, 'firm', 'year', homogeneous=True)
res_hom = sur_hom.fit()

print("Homogeneous SUR (pooled slopes):")
print(res_hom.params)
print(f"\nSystem R-squared (unrestricted): {res_unr.system_rsquared:.4f}")
print(f"System R-squared (homogeneous):  {res_hom.system_rsquared:.4f}")

In [ ]:
# LR test for slope homogeneity
lr = sur_unr.lr_test_homogeneity()

print("Likelihood Ratio Test for Slope Homogeneity")
print("=" * 45)
print(f"H0: beta_1 = beta_2 = ... = beta_N")
print(f"LR statistic: {lr['statistic']:.4f}")
print(f"p-value:      {lr['pvalue']:.6f}")
print(f"df:           {lr['df']}")
print()
if lr['reject_h0']:
    print("Reject H0: Slopes differ significantly across entities.")
    print("Unrestricted SUR is preferred.")
else:
    print("Fail to reject H0: Homogeneous slopes are plausible.")
    print("Homogeneous SUR may be more efficient.")

In [ ]:
# Compare unrestricted entity-specific coefficients with homogeneous
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, var in enumerate(['value', 'capital']):
    ax = axes[idx]
    coefs = [res_unr.entity_params[e][var] for e in entities]
    
    ax.barh(range(len(entities)), coefs, color='steelblue', alpha=0.7, edgecolor='black')
    
    if var in res_hom.params.index:
        hom_val = res_hom.params[var]
        ax.axvline(hom_val, color='red', linestyle='--', linewidth=2,
                   label=f'Homogeneous = {hom_val:.4f}')
    
    ax.set_yticks(range(len(entities)))
    ax.set_yticklabels([f'Firm {e}' for e in entities])
    ax.set_xlabel('Coefficient')
    ax.set_title(f'SUR Coefficients: {var}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Unrestricted vs Homogeneous SUR', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Application: Credit Risk Pools

Consider 4 credit risk pools (e.g., mortgage, auto, credit card, student loans) observed over 40 quarters. Default rates in each pool respond to macroeconomic conditions, but the sensitivities differ across pools.

We simulate data where:
- Errors are correlated across pools (systemic risk)
- Each pool has different covariates

In [ ]:
np.random.seed(123)
N_pools = 4
T_periods = 40
pool_names = ['Mortgage', 'Auto', 'CreditCard', 'StudentLoan']

# True cross-pool error correlation
true_corr = np.array([
    [1.0, 0.6, 0.3, 0.2],
    [0.6, 1.0, 0.5, 0.3],
    [0.3, 0.5, 1.0, 0.7],
    [0.2, 0.3, 0.7, 1.0],
])
true_sigma_diag = np.array([2.0, 1.5, 1.0, 0.8])
true_sigma = np.outer(true_sigma_diag, true_sigma_diag) * true_corr

# Generate correlated errors
L = np.linalg.cholesky(true_sigma)
eps = (L @ np.random.randn(N_pools, T_periods)).T  # (T, N)

# Covariates
gdp_growth = np.random.normal(2, 1, T_periods)
unemployment = np.random.normal(5, 1.5, T_periods)
interest_rate = np.random.normal(3, 0.5, T_periods)
consumer_conf = np.random.normal(100, 10, T_periods)

# True coefficients (different per pool)
true_betas = {
    'Mortgage':    {'const': 5, 'gdp': -0.8, 'unemp': 0.5, 'rate': 0.3},
    'Auto':        {'const': 3, 'gdp': -0.5, 'unemp': 0.3},
    'CreditCard':  {'const': 7, 'unemp': 0.8, 'consumer': -0.04},
    'StudentLoan': {'const': 4, 'gdp': -0.3, 'unemp': 0.6},
}

rows = []
for t in range(T_periods):
    for j, pool in enumerate(pool_names):
        betas = true_betas[pool]
        y = betas['const']
        if 'gdp' in betas:
            y += betas['gdp'] * gdp_growth[t]
        if 'unemp' in betas:
            y += betas['unemp'] * unemployment[t]
        if 'rate' in betas:
            y += betas['rate'] * interest_rate[t]
        if 'consumer' in betas:
            y += betas['consumer'] * consumer_conf[t]
        y += eps[t, j]
        rows.append({
            'pool': pool, 'quarter': t + 1,
            'default_rate': y,
            'gdp_growth': gdp_growth[t],
            'unemployment': unemployment[t],
            'interest_rate': interest_rate[t],
            'consumer_conf': consumer_conf[t],
        })

credit_data = pd.DataFrame(rows)
print(f"Simulated credit data: {credit_data.shape}")
credit_data.head(8)

In [ ]:
# SUR with heterogeneous formulas
credit_formulas = {
    'Mortgage':    'default_rate ~ gdp_growth + unemployment + interest_rate',
    'Auto':        'default_rate ~ gdp_growth + unemployment',
    'CreditCard':  'default_rate ~ unemployment + consumer_conf',
    'StudentLoan': 'default_rate ~ gdp_growth + unemployment',
}

sur_credit = pb.SUR(credit_formulas, credit_data, 'pool', 'quarter')
res_credit = sur_credit.fit()

print(res_credit.system_summary())

In [ ]:
# Compare estimated vs true correlations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# True correlation
im0 = axes[0].imshow(true_corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(N_pools))
axes[0].set_yticks(range(N_pools))
axes[0].set_xticklabels(pool_names, rotation=45, ha='right')
axes[0].set_yticklabels(pool_names)
axes[0].set_title('True Correlation')
for i in range(N_pools):
    for j in range(N_pools):
        axes[0].text(j, i, f'{true_corr[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im0, ax=axes[0])

# Estimated correlation
res_credit.plot_correlation_matrix(ax=axes[1])
axes[1].set_title('Estimated Correlation (SUR)')

plt.tight_layout()
plt.show()

In [ ]:
# Compare estimated coefficients with true values
print(f"{'Pool':<14} {'Variable':<16} {'True':>8} {'SUR':>8} {'Error':>8}")
print("-" * 58)
for pool in pool_names:
    params = res_credit.entity_params[pool]
    betas = true_betas[pool]
    for var in params.index:
        if var == 'const':
            true_val = betas.get('const', 0)
        elif var == 'gdp_growth':
            true_val = betas.get('gdp', 0)
        elif var == 'unemployment':
            true_val = betas.get('unemp', 0)
        elif var == 'interest_rate':
            true_val = betas.get('rate', 0)
        elif var == 'consumer_conf':
            true_val = betas.get('consumer', 0)
        else:
            true_val = 0
        est_val = params[var]
        print(f"{pool:<14} {var:<16} {true_val:>8.3f} {est_val:>8.3f} {est_val - true_val:>8.3f}")

The SUR recovers the true coefficients well, and the estimated cross-pool correlations match the true DGP.

---
## 10. Exercises

### Exercise 1: SUR with Simulated Independent Errors

Simulate a panel with N=5 entities, T=50 periods, and **independent errors** ($\Sigma$ is diagonal). Fit SUR and check:
- Does the BP test reject independence?
- Are the efficiency gains close to 1.0?

**Hint:** Generate errors as `np.random.normal(0, 1, (T, N))` without cross-correlation.

In [ ]:
# Your code here


### Exercise 2: ISUR Convergence Speed

Using the Grunfeld data, fit ISUR with different tolerances: `tol=1e-4`, `tol=1e-8`, `tol=1e-12`. How many iterations does each need? Do the coefficients change meaningfully?

**Hint:** Compare `res.n_iterations` and `res.entity_params` across the three fits.

In [ ]:
# Your code here


### Exercise 3: Homogeneity Test with Homogeneous DGP

Simulate a panel where all entities truly share the same slope ($\beta_i = 2.0$ for all $i$), but errors are correlated. Fit unrestricted SUR and run the LR homogeneity test. Does it (correctly) fail to reject?

**Hint:** Use `sur.lr_test_homogeneity()` and check the p-value.

In [ ]:
# Your code here


---

**Solutions** are available in `solutions/02_sur_solutions.ipynb`.